# SGD regularizer grid on the tiny circuit

The low-data tiny-circuit task from `tiny_circuit.ipynb` (5 wires, depth 3,
circuit seed 5, solo wire 0, `train_frac=0.5`) trained with **SGD +
momentum** instead of Adam, sweeping **all combinations of `weight_decay`
and `weight_noise`** over the 7-shape model-size grid.

SGD removes the adaptive-optimizer confounds (slingshot spikes, second-
moment feedback from injected noise), so what survives here is task /
regularizer phenomenology, not Adam artifacts. The Adam LR table does not
transfer to SGD, so a short LR-tune stage runs first (on the clean
configuration; the tuned LR is then held fixed across the grid).

Stages: LR-tune (7 shapes x 4 LRs x 1k steps) -> main grid (7 shapes x
3 wd x 3 wn x 10k steps = 63 runs) -> one big trajectory figure + a
summary figure. Everything is idempotent and resumable.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import os
    %pip -q install -U "jax[cuda12]" optax
    if not os.path.exists("/content/circscale"):
        !git clone https://github.com/amdson/circscale.git /content/circscale
    %cd /content/circscale
    !git pull
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/circscale_runs", exist_ok=True)
    if not os.path.islink("runs"):
        os.symlink("/content/drive/MyDrive/circscale_runs", "runs")

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from train import RunConfig, load_run, run

# --- task (matches tiny_circuit.ipynb's low-data section) ---
N_WIRES, CIRC_DEPTH, CIRCUIT_SEED, TARGET_WIRE = 5, 3, 5, 0
TRAIN_FRAC = 0.5

# --- sweep ---
SHAPES = [(32, 2), (64, 3), (128, 4), (180, 5), (256, 6), (360, 7), (512, 8)]
WD_GRID = [0.0, 1e-2, 1e-1]
WN_GRID = [0.0, 0.03, 0.1]  # init-relative: fraction of each layer's init std
MOMENTUM = 0.9
STEPS = 10_000
LR_GRID = [0.03, 0.1, 0.3, 1.0]
TUNE_STEPS = 1_000
OUT_DIR = "runs/tiny5_sgd"
CHANCE = np.log(2)


def base_cfg(width, depth, lr, **kw):
    return RunConfig(width=width, mlp_depth=depth, lr=lr,
                     optimizer="sgd", momentum=MOMENTUM,
                     n_wires=N_WIRES, circ_depth=CIRC_DEPTH,
                     circuit_seed=CIRCUIT_SEED, output_wires=(TARGET_WIRE,),
                     train_frac=TRAIN_FRAC, eval_every=100,
                     out_dir=OUT_DIR, **kw)


def n_params(w, d, hr=4):
    h = hr * w
    return N_WIRES * w + d * (w + w * h + h * w) + w + w * N_WIRES

## LR tune

Short clean runs (`wd = wn = 0`) per shape; picks the LR with the lowest
train-pool BCE after 1k steps (optimization speed, not generalization —
the criterion a fixed-LR comparison should hold constant). Divergent runs
(NaN) are treated as infinitely bad. Warns when the best LR sits at a grid
edge.

In [ ]:
tuned_lr = {}
for w, d in SHAPES:
    losses = {}
    for lr in LR_GRID:
        cfg = base_cfg(w, d, lr, steps=TUNE_STEPS)
        run(cfg)
        loss = load_run(cfg.npz_path)[1]["per_out_loss_tr"][-1, TARGET_WIRE]
        losses[lr] = np.nan_to_num(loss, nan=np.inf)
    tuned_lr[(w, d)] = best = min(losses, key=losses.get)
    edge = "  ** best at grid edge — extend LR_GRID **" \
        if best in (LR_GRID[0], LR_GRID[-1]) else ""
    print(f"w{w}d{d}: best lr={best:g}   " +
          "  ".join(f"{lr:g}: {l:.4f}" for lr, l in losses.items()) + edge)

## Main grid (idempotent — interrupt and re-run freely)

7 shapes x 3 weight decays x 3 noise levels at each shape's tuned LR.

In [ ]:
grid_cfgs = {
    (w, d, wd, wn): base_cfg(w, d, tuned_lr[(w, d)], steps=STEPS,
                             weight_decay=wd, weight_noise=wn)
    for w, d in SHAPES for wd in WD_GRID for wn in WN_GRID
}
print(f"{len(grid_cfgs)} runs")
for cfg in grid_cfgs.values():
    run(cfg)

## The big figure

Rows: model shapes. Columns: (weight_decay, weight_noise) combinations.
Each panel: wire-0 BCE on the 16 training inputs (dashed) vs the 16
held-out inputs (solid), log-log; chance dotted. Losses clipped at 1e-6
for the log axis.

In [ ]:
res = {k: load_run(c.npz_path)[1]
       for k, c in grid_cfgs.items() if c.npz_path.exists()}
combos = [(wd, wn) for wd in WD_GRID for wn in WN_GRID]

fig, axes = plt.subplots(len(SHAPES), len(combos),
                         figsize=(2.4 * len(combos), 2.0 * len(SHAPES)),
                         sharex=True, sharey=True, squeeze=False)
for i, (w, d) in enumerate(SHAPES):
    for j, (wd, wn) in enumerate(combos):
        ax = axes[i, j]
        r = res.get((w, d, wd, wn))
        if r is None:
            ax.axis("off")
            continue
        m = r["eval_steps"] > 0
        steps = r["eval_steps"][m]
        tr = np.maximum(r["per_out_loss_tr"][m][:, TARGET_WIRE], 1e-6)
        ho = np.maximum(r["per_out_loss_ho"][m][:, TARGET_WIRE], 1e-6)
        ax.plot(steps, tr, "C0--", lw=1.0, label="train")
        ax.plot(steps, ho, "C1-", lw=1.0, label="held-out")
        ax.axhline(CHANCE, color="gray", ls=":", lw=0.6)
        ax.set(xscale="log", yscale="log")
        if i == 0:
            ax.set_title(f"wd={wd:g}\nwn={wn:g}", fontsize=8)
        if j == 0:
            ax.set_ylabel(f"w{w}d{d}", fontsize=8)
axes[0, 0].legend(fontsize=6)
fig.supxlabel("step")
fig.supylabel(f"wire {TARGET_WIRE} BCE")
plt.tight_layout()

## Summary

Final held-out BCE and accuracy vs parameter count, one line per (wd, wn)
combo: color = noise level, line style = weight decay. Accuracy has
granularity 1/16.

In [ ]:
Ns = [n_params(w, d) for w, d in SHAPES]
wd_ls = dict(zip(WD_GRID, ["-", "--", ":"]))
wn_col = dict(zip(WN_GRID, plt.cm.viridis(np.linspace(0, 0.85, len(WN_GRID)))))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.4))
for wd, wn in combos:
    bce = [res[(w, d, wd, wn)]["per_out_loss_ho"][-1, TARGET_WIRE]
           if (w, d, wd, wn) in res else np.nan for w, d in SHAPES]
    acc = [res[(w, d, wd, wn)]["per_out_acc_ho"][-1, TARGET_WIRE]
           if (w, d, wd, wn) in res else np.nan for w, d in SHAPES]
    kw = dict(ls=wd_ls[wd], color=wn_col[wn], marker="o", ms=3,
              label=f"wd={wd:g} wn={wn:g}")
    ax1.plot(Ns, bce, **kw)
    ax2.plot(Ns, acc, **kw)
ax1.axhline(CHANCE, color="gray", ls=":", lw=0.8)
ax1.set(xscale="log", yscale="log", xlabel="params N",
        ylabel="final held-out BCE")
ax2.axhline(0.5, color="gray", ls=":", lw=0.8)
ax2.set(xscale="log", xlabel="params N", ylabel="final held-out acc",
        ylim=(0.4, 1.02))
ax1.legend(fontsize=6, ncol=3)
plt.tight_layout()

hdr = " " * 8 + "".join(f"wd{wd:g}/wn{wn:g}".rjust(14) for wd, wn in combos)
print("final held-out acc (wire 0)\n" + hdr)
for w, d in SHAPES:
    row = "".join(
        f"{res[(w, d, wd, wn)]['per_out_acc_ho'][-1, TARGET_WIRE]:>14.3f}"
        if (w, d, wd, wn) in res else f"{'—':>14s}"
        for wd, wn in combos)
    print(f"{f'w{w}d{d}':>8s}{row}")

## Notes

- LR is tuned once per shape on the clean configuration and held fixed
  across the (wd, wn) grid — noisy runs might individually prefer a lower
  LR, so read strong-noise instability as "at the clean-tuned LR".
- `weight_noise` here is the transient (ELBO-style) mode in init-relative
  units: each leaf gets std `wn x (its init std)`, so the function-space
  perturbation is comparable across the 7 widths (norm scales get no
  noise). Add `noise_mode="persist"` configs to compare Langevin-style
  noise, or `noise_scale="abs"` for raw per-parameter stds. With SGD,
  `weight_decay` is classic L2-coupled decay.
- With the 16-input pool and batch 256, each batch is ~16 copies of the
  whole pool, so gradients are near-exact: weight noise is essentially the
  only stochasticity in these runs.
- Momentum is 0.9 throughout; set `MOMENTUM = 0.0` for fully vanilla SGD
  (run names then pick up an `m0` tag, so both sets can coexist).
- Grids are module constants — extend `WD_GRID` / `WN_GRID` and re-run;
  completed runs are skipped.